# Depth-Conditioned Multi-Context V_θ — OpenWebText Scale-Up (Fock-PARFLM v2.1)

## Motivation

The multi-context V_θ notebook (`colab_fock_multicontext_vtheta_openwebtext.ipynb`)
gave each xi channel its own Gaussian well bank, curing the **functional-
diversity deficit across temporal horizons**.  But all `L` Verlet layers
still share *one* V_θ instance: the same well landscape is applied at every
integration step.  Early layers (coarse, embedding-near states) and late
layers (refined, output-near states) are forced to use identical attractors.

Fully untying V_θ across layers (`G = L` independent banks) fixes this but
is expensive: the well-bank projections dominate the parameter count, so
`G = 16` inflates V_θ from ~12M to ~190M params (a ~230M total model that is
no longer a fair small-GPT-2 comparison, and whose extra params buy *only*
self-energy shape, not context-mixing capacity).

## The fix: depth-conditioning (option A)

Keep a **single shared** multi-context well bank and give each layer a small
learned **depth code** `e_g ∈ R^{n_ctx × d}` added to the per-channel context
ξ before the bank projections:

> ξ_g^(m) = ξ^(m) + e_g^(m),    V_g(xis, h) = Σ_m V_m(ξ_g^(m), h)

Each layer sees a *shifted* view of the same wells → a distinct effective
potential per layer, at the cost of only `L · n_ctx · d ≈ 31k` extra params
(vs ~178M for full untying). Cost collapses from ~190M back to ~12M (G=1),
keeping the model at GPT-2-Small-class size while still testing the
per-layer-specialisation hypothesis.

- **Conservativity**: `e_g` is constant w.r.t. `h`, so the per-step force is
  still `−∇_h V` of a scalar potential. Per-step conservativity preserved.
- **Limitation**: shift-only — the well *shapes* are tied across layers, only
  the conditioning differs. If shift-only proves too weak, the next rung is
  LoRA-across-depth (shared backbone + per-layer rank-r delta, ~21M at G=16).

## What changes vs the multi-context V_θ notebook

### 1. Depth-conditioned multi-context V_θ

`DepthConditionedMultiContextGaussianVTheta` wraps one
`MultiContextGaussianVTheta` bank with a `(L, n_ctx, d)` depth-code table.
The model's per-layer step broadcasts the current layer index to the bank
via `install_depth_routing(model)` (safe under gradient checkpointing — the
layer pointer is set synchronously inside each layer-step call).

### A/B comparison toggle

This notebook runs **either arm** of a controlled depth-conditioning study
via a single flag in Cell 0:

| `V_THETA_DEPTH_CONDITION` | Arm | V_θ used | Variant tag | Question answered |
|---|---|---|---|---|
| `True`  | **A (treatment)** | depth-conditioned shared bank + per-layer codes | `dcvt…` | Does per-layer specialisation help? |
| `False` | **B (control)**   | plain shared multi-context bank (one V_θ, all layers) | `mcvt…` | baseline without per-layer codes |

Both arms are otherwise **identical** (same xi channels, K wells, WSD
schedule, V_φ widths, batch/accum, steps, seed), so the only difference is
the `≈ L·n_ctx·d` depth-code parameters and the per-layer routing.

The two arms write to **distinct** Drive/checkpoint paths automatically
because the variant tag (`dcvt…` vs `mcvt…`) is baked into
`_gdrive_name`, `CKPT_PREFIX`, and the local results dir — so you can run
A and B back-to-back (or in parallel sessions) without clobbering each
other's checkpoints or logs. To run the control, set
`V_THETA_DEPTH_CONDITION = False` and re-run all cells.

### 2. WSD learning-rate schedule

Replaces cosine decay with Warmup-Stable-Decay (WSD):
- **Warmup** (first `WSD_WARMUP_FRAC` of steps): linear ramp to peak LR.
- **Stable** (next `WSD_STABLE_FRAC` of steps): hold at peak LR.
- **Decay** (remaining): cosine anneal from peak LR to `WSD_LR_FLOOR`.

This keeps the model at peak learning rate through the middle of
training, giving the multi-context wells more time at high LR to
separate and specialise before the final anneal.

### 3. Wider type/angle subspaces + routing fan-out

- `V_PHI_D_TYPE` doubled from 16 to 32 — richer "who should interact"
  discrimination per V_φ head.
- `V_PHI_D_ANGLE` doubled from 8 to 16 — more directional force
  diversity per pair interaction.
- `TOP_K` doubled from 8 to 16 — each token routes to twice as many
  past tokens through the pair potential (zero param cost, ~2× sparse
  routing FLOPs).

### 4. Explicit batch/accum and longer training

- `GRAD_ACCUM` is a first-class config knob (not auto-probed).
- `TOTAL_STEPS` defaults to 300K (50% more than the 200K baseline).
- Peak `LR` raised to 3e-4 (trainable with WSD and proper warmup).

## Knobs (Cell 0)

All knobs from the multi-head V_φ notebook are preserved.  New additions:

| Knob | Meaning |
|------|---------|
| `V_THETA_DEPTH_CONDITION` | **new** — enable per-layer depth codes on a shared bank |
| `V_THETA_DEPTH_CODE_INIT_STD` | **new** — init std of the depth-code table (default 0.02) |
| `V_THETA_N_HEADS` | per-xi-channel V_θ heads (default = XI_CHANNELS) |
| `V_THETA_WELLS_PER_HEAD` | Gaussian wells per head (default 8) |
| `LR_SCHEDULE` | `'cosine'` or `'wsd'` |
| `WSD_WARMUP_FRAC` | **new** — fraction of steps for warmup (default 0.05) |
| `WSD_STABLE_FRAC` | **new** — fraction of steps at peak LR (default 0.60) |
| `WSD_LR_FLOOR` | **new** — minimum LR at end of decay (default LR * 0.05) |
| `V_PHI_D_TYPE` | **new** — type-vector dimension per V_φ head (default 32, was 16) |
| `V_PHI_D_ANGLE` | **new** — angle-vector dimension per V_φ head (default 16, was 8) |
| `TOP_K` | **changed** — 16 (was 8); doubles pair-interaction fan-out |
| `GRAD_ACCUM` | **changed** — explicit config instead of auto-probed |

## Prerequisites

- OpenWebText tokenized and cached on Google Drive (reused from earlier phases)
- H100 80 GB GPU


In [1]:
# ── Cell 0: Configuration ─────────────────────────────────────────

# ── V_theta architecture ────────────────────────────────────────────
# V_THETA_VARIANT selects the potential well family:
#   'gaussian'  — MixtureGaussianVTheta (bounded, K wells)
#   'sarf'      — SARFGaussianVTheta (bounded, frozen PMI anchors)
#   'sq3'       — MixtureQuadraticVTheta (unbounded, K wells)
#   'mlp'       — keep the default MLP V_theta (unstructured baseline)
V_THETA_VARIANT = 'gaussian'    # 'gaussian', 'sarf', 'sq3', or 'mlp'
W_SCALE         = 1.0
BG_QUAD_EPS     = 0.0
SQ3_TAU         = 1.0
SQ3_CURV_MAX    = 2.0
SARF_N_ANCHORS  = 64

# ── Multi-context V_theta heads (Bottleneck-2 cure) ──────────────
# V_THETA_N_HEADS: number of per-xi-channel well banks.  Set to
#   XI_CHANNELS (default) for one bank per xi horizon.  Set to 1 for
#   the concat baseline (original GaussianVThetaMultiXiAdapter).
# V_THETA_WELLS_PER_HEAD: Gaussian wells per bank.  With 5 heads and
#   8 wells/head the model gets 40 total attractors, each conditioned
#   on a single temporal resolution, vs 8 attractors on a blurred
#   K*d concat in the baseline.  Total param cost ≈ baseline.
V_THETA_WELLS_PER_HEAD = 8      # wells per multi-context head
# V_THETA_N_HEADS is set after XI_CHANNELS is resolved below.

# ── Depth-conditioning (option A: cheap per-layer untying) ───────
# V_THETA_DEPTH_CONDITION: when True, a SINGLE shared multi-context well
#   bank is shared across all L Verlet layers, but each layer adds a small
#   learned depth code e_g in R^{n_ctx x d} to xi before the bank
#   projections.  This gives each layer a distinct effective potential at
#   the cost of only L * n_ctx * d extra params (~31k at L=16), instead of
#   replicating the ~12M bank L times (~190M for G=16).
#   Requires V_THETA_N_HEADS > 1 and V_THETA_VARIANT == 'gaussian'.
# V_THETA_DEPTH_CODE_INIT_STD: init std of the depth-code table.  Small so
#   the model starts ~tied (all layers share one bank) and learns to
#   differentiate.
# A/B toggle for the controlled study:
#   True  -> Arm A (treatment): depth-conditioned (tag 'dcvt...')
#   False -> Arm B (control):   plain multi-context bank (tag 'mcvt...')
# The two arms write to distinct Drive/ckpt paths (tag is in the prefix),
# so they never clobber each other.  Everything else is held identical.
V_THETA_DEPTH_CONDITION     = True   # per-layer depth codes on a shared bank
V_THETA_DEPTH_CODE_INIT_STD = 0.02   # init std of the (L, n_ctx, d) code table

# ── Xi channel override ──────────────────────────────────────────────
XI_OVERRIDE     = '5long'       # '5long' (default) | 5 | 6 | '4long' | None

_XI_PRESETS_CFG = {
    5:       [0.25, 0.50, 0.75, 0.95, 0.99],
    '5long': [0.50, 0.75, 0.95, 0.99, 0.995],
    6:       [0.25, 0.50, 0.75, 0.95, 0.99, 0.995],  # keeps short channel, adds 200-tok horizon
    '4long': [0.50, 0.75, 0.95, 0.995],
}
if XI_OVERRIDE is None:
    XI_ALPHA_INITS = [0.25, 0.50, 0.75, 0.95]
elif XI_OVERRIDE in _XI_PRESETS_CFG:
    XI_ALPHA_INITS = _XI_PRESETS_CFG[XI_OVERRIDE]
else:
    raise ValueError(f'Unsupported XI_OVERRIDE={XI_OVERRIDE!r}; use None, 5, 6, "5long", or "4long"')
XI_CHANNELS = len(XI_ALPHA_INITS)

# Now that XI_CHANNELS is known, set V_THETA_N_HEADS default.
# Override to 1 for the concat baseline comparison.
V_THETA_N_HEADS = XI_CHANNELS    # one bank per xi channel (default)

# ── PARF V_phi knobs ─────────────────────────────────────────────
V_PHI_KIND      = 'structural_competitive'  # 'structural_competitive' | 'structural' | 'mlp'
V_PHI_MLP_HIDDEN = 128
TOP_K           = 16            # sparse pairs per query (was 8; 16 doubles context coverage)
V_PHI_N_HEADS   = 4
# Type/angle subspace widths: control V_phi's per-pair expressivity.
# With H multi-head V_phi heads, effective type-space is H * V_PHI_D_TYPE
# and effective angle-space is H * V_PHI_D_ANGLE.  Doubling from the
# defaults (16/8) is cheap in params and lifts the directional-force
# diversity ceiling.
V_PHI_D_TYPE    = 32            # type-vector dimension d_l (default 16)
V_PHI_D_ANGLE   = 16            # value-angle dimension K  (default 8)

# ── Fock reverse-channel stabilisation (E5c; see §10.12 of
#    Improving_the_Fock_Mechanism_to_match_Attention.md) ───────────
# The reverse channel is the sole non-conservative force and was the
# dominant gradient-spike source on OWT.  These knobs bound it without
# removing the directed-routing benefit:
#   REVERSE_CHANNEL              — master on/off (E5a sets this False).
#   REVERSE_CHANNEL_STABLE       — QK-norm + output RMS-norm readout.
#   REVERSE_CHANNEL_PRE_LN       — pre-LayerNorm on q/k/v (stable only).
#   REVERSE_CHANNEL_SOFT_NORM    — soft-floored output norm (eps=1.0)
#       instead of hard unit-RMS (eps=1e-6).  The hard norm pins the force
#       to unit RMS but its 1/||Q|| Jacobian inflates the PRE-clip
#       projection gradient when the natural force is small (the benign
#       'override:reverse_ch' spike).  The soft floor divides by
#       sqrt(mean(Q^2)+1): identity for small forces (Jacobian ~1) and a
#       soft cap at ~unit RMS for large ones, so the gradient stays O(1)
#       and the 0.1 clip rarely binds (the reverse channel can learn
#       faster).  Stable-only; harmless to leave True.
#   REVERSE_CHANNEL_WARMUP_STEPS — linear gate warmup over N forward
#       passes (0 = off).  With GRAD_ACCUM=2 a forward ≈ half an opt step,
#       so 4000 forwards ≈ 2000 optimiser steps of ramp.
REVERSE_CHANNEL              = True
REVERSE_CHANNEL_STABLE      = True
REVERSE_CHANNEL_PRE_LN      = True
REVERSE_CHANNEL_SOFT_NORM   = True
REVERSE_CHANNEL_WARMUP_STEPS = 4000
# One-time transitional reset: when resuming a PRE-stable checkpoint into
# the stabilised reverse channel, the loaded reverse_channel_scale carries
# a value learned under the unnormalised readout (e.g. -0.227).  Set this
# True for that first stable restart to zero the gate (and warmup) so the
# bounded force opens cleanly under the ramp; set back to False afterwards
# so later resumes keep the learned gate.
REVERSE_CHANNEL_RESET_SCALE  = False

# ── Output read-out head (D0.4 long-tail fix) ────────────────────
USE_OUTPUT_BIAS = True
TIE_EMBEDDINGS  = False         # untied W_out for this experiment

# ── Optimizer ─────────────────────────────────────────────────────
OPTIMIZER = 'adamw'             # 'adamw' | 'lamb' | 'lion'
GRAD_CENTRALIZATION = False

# ── LR schedule ──────────────────────────────────────────────────
# 'cosine': standard cosine anneal (same as multi-head notebook).
# 'wsd':    Warmup-Stable-Decay — holds peak LR through the middle
#           of training, giving wells more time to specialise.
LR_SCHEDULE     = 'wsd'         # 'cosine' | 'wsd'
WSD_WARMUP_FRAC = 0.05          # fraction of TOTAL_STEPS for linear warmup
WSD_STABLE_FRAC = 0.60          # fraction at peak LR (plateau phase)
WSD_LR_FLOOR    = None          # set below after LR is known; default = LR * 0.05

# ── Batch / accumulation (explicit, not auto-probed) ─────────────
GRAD_ACCUM      = 2             # gradient accumulation steps
# BATCH_SIZE is still auto-probed for OOM safety, but GRAD_ACCUM is
# fixed so the effective batch size is predictable.

# ── Variant tag ──────────────────────────────────────────────────
_variant_parts = []
if V_THETA_VARIANT == 'mlp':
    _variant_parts.append('mlp_vtheta')
elif V_THETA_VARIANT == 'sq3':
    _variant_parts.append(f'sq3_k{V_THETA_WELLS_PER_HEAD}')
elif V_THETA_VARIANT == 'gaussian' and V_THETA_WELLS_PER_HEAD != 8:
    _variant_parts.append(f'k{V_THETA_WELLS_PER_HEAD}')
if V_PHI_KIND == 'mlp':
    _variant_parts.append(f'mlp_vphi_h{V_PHI_MLP_HIDDEN}')
elif V_PHI_KIND == 'structural':
    _variant_parts.append('struct_vphi')
if XI_OVERRIDE is not None:
    _variant_parts.append(f'xi{XI_OVERRIDE}')
    # int 5 → 'xi5', int 6 → 'xi6', str '5long' → 'xi5long'
if TOP_K != 8:
    _variant_parts.append(f'topk{TOP_K}')
if V_PHI_D_TYPE != 16 or V_PHI_D_ANGLE != 8:
    _variant_parts.append(f'dt{V_PHI_D_TYPE}da{V_PHI_D_ANGLE}')
if V_PHI_N_HEADS != 1:
    _variant_parts.append(f'mh{V_PHI_N_HEADS}')
if V_THETA_N_HEADS > 1:
    if V_THETA_DEPTH_CONDITION:
        _variant_parts.append(f'dcvt{V_THETA_N_HEADS}x{V_THETA_WELLS_PER_HEAD}')
    else:
        _variant_parts.append(f'mcvt{V_THETA_N_HEADS}x{V_THETA_WELLS_PER_HEAD}')
if USE_OUTPUT_BIAS:
    _variant_parts.append('ob')
if not TIE_EMBEDDINGS:
    _variant_parts.append('untied')
if OPTIMIZER != 'adamw':
    _variant_parts.append(OPTIMIZER)
if GRAD_CENTRALIZATION:
    _variant_parts.append('gc')
if LR_SCHEDULE != 'cosine':
    _variant_parts.append(LR_SCHEDULE)
# Reverse-channel experiment arm -> its OWN run dir, so the stabilised
# variant co-adapts from step 0 instead of retrofitting onto a checkpoint
# that converged without it (which diverges; see E5c notes).  'e5c' =
# stable reverse channel on from scratch; 'e5a' = reverse channel off.
if not REVERSE_CHANNEL:
    _variant_parts.append('e5a')
elif REVERSE_CHANNEL_STABLE:
    _variant_parts.append('e5c')
_variant_tag = '_'.join(_variant_parts)

print(f'Config: V_theta={V_THETA_VARIANT}')
if V_THETA_VARIANT == 'mlp':
    print(f'  V_theta=MLP (unstructured)')
elif V_THETA_VARIANT == 'gaussian':
    print(f'  wells_per_head={V_THETA_WELLS_PER_HEAD}, w_scale={W_SCALE}')
elif V_THETA_VARIANT == 'sq3':
    print(f'  wells_per_head={V_THETA_WELLS_PER_HEAD}, tau={SQ3_TAU}, curvature_max={SQ3_CURV_MAX}')
else:
    print(f'  SARF N_S={SARF_N_ANCHORS}, w_scale={W_SCALE}')
if V_THETA_N_HEADS > 1:
    print(f'  multi-context V_theta: {V_THETA_N_HEADS} heads x {V_THETA_WELLS_PER_HEAD} wells = '
          f'{V_THETA_N_HEADS * V_THETA_WELLS_PER_HEAD} total attractors')
    if V_THETA_DEPTH_CONDITION:
        print(f'  depth-conditioned: shared bank + per-layer codes '
              f'(init_std={V_THETA_DEPTH_CODE_INIT_STD})')
else:
    print(f'  concat baseline V_theta: K={V_THETA_WELLS_PER_HEAD}')
print(f'  V_phi={V_PHI_KIND}'
      + (f' (mlp_hidden={V_PHI_MLP_HIDDEN})' if V_PHI_KIND == 'mlp' else ''))
if XI_OVERRIDE is not None:
    print(f'  XI_OVERRIDE={XI_OVERRIDE} -> {XI_CHANNELS}ch, '
          f'horizons ~{[round(1/(1-a),1) for a in XI_ALPHA_INITS]} tok')
print(f'  PARF top_k={TOP_K}, V_phi heads={V_PHI_N_HEADS}, '
      f'd_type={V_PHI_D_TYPE} (eff {V_PHI_N_HEADS*V_PHI_D_TYPE}), '
      f'd_angle={V_PHI_D_ANGLE} (eff {V_PHI_N_HEADS*V_PHI_D_ANGLE})')
print(f'  read-out: output_bias={USE_OUTPUT_BIAS}, '
      + ('tied E^T' if TIE_EMBEDDINGS else 'UNTIED W_out'))
print(f'  optimizer={OPTIMIZER}  grad_centralization={GRAD_CENTRALIZATION}')
print(f'  LR schedule={LR_SCHEDULE}  grad_accum={GRAD_ACCUM}')
if _variant_tag:
    print(f'  [variant] tag={_variant_tag}')


Config: V_theta=gaussian
  wells_per_head=8, w_scale=1.0
  multi-context V_theta: 5 heads x 8 wells = 40 total attractors
  depth-conditioned: shared bank + per-layer codes (init_std=0.02)
  V_phi=structural_competitive
  XI_OVERRIDE=5long -> 5ch, horizons ~[2.0, 4.0, 20.0, 100.0, 200.0] tok
  PARF top_k=16, V_phi heads=4, d_type=32 (eff 128), d_angle=16 (eff 64)
  read-out: output_bias=True, UNTIED W_out
  optimizer=adamw  grad_centralization=False
  LR schedule=wsd  grad_accum=2
  [variant] tag=xi5long_topk16_dt32da16_mh4_dcvt5x8_ob_untied_wsd_e5c


In [2]:
# ── Cell 1: Environment ───────────────────────────────────────────
import os, sys, gc, shutil, subprocess, json, time, math
from pathlib import Path
from dataclasses import asdict

os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

REPO_URL    = 'https://github.com/dimitarpg13/semsimula-paper.git'
REPO_BRANCH = 'main'

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}')


def _sh(cmd):
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'exit {r.returncode}: {cmd}')


if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    REPO_ROOT = Path('/content/semsimula-paper')
    if not (REPO_ROOT / '.git').exists():
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        _sh(f'git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_ROOT}')
    else:
        try:
            _sh(f'git -C {REPO_ROOT} fetch --depth 1 origin {REPO_BRANCH}')
            _sh(f'git -C {REPO_ROOT} reset --hard origin/{REPO_BRANCH}')
        except RuntimeError as e:
            print(f'WARNING: repo refresh failed ({e}); using existing checkout.')

    _gdrive_name = 'semsimula_fock_depthcond_vtheta_owt'
    if _variant_tag:
        _gdrive_name += f'_{_variant_tag}'
    GDRIVE_ROOT = Path(f'/content/drive/MyDrive/{_gdrive_name}')
    GDRIVE_ROOT.mkdir(parents=True, exist_ok=True)

    DATA_DIR = GDRIVE_ROOT / 'data'
    DATA_DIR.mkdir(exist_ok=True)
    repo_data = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    if repo_data.is_symlink():
        repo_data.unlink()
    elif repo_data.is_dir():
        shutil.rmtree(repo_data)
    repo_data.symlink_to(DATA_DIR)

    CKPT_DIR    = GDRIVE_ROOT / 'checkpoints'
    RESULTS_DIR = GDRIVE_ROOT / 'results'
    CKPT_DIR.mkdir(exist_ok=True)
    RESULTS_DIR.mkdir(exist_ok=True)

    _sh('pip install -q transformers huggingface_hub pyarrow')
else:
    REPO_ROOT = Path('.').resolve()
    while not (REPO_ROOT / '.git').exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent
    DATA_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    _local_phase = 'depthcond_vtheta' + (f'_{_variant_tag}' if _variant_tag else '')
    CKPT_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup' / 'results' / _local_phase / 'ckpts'
    RESULTS_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup' / 'results' / _local_phase
    for d in [DATA_DIR, CKPT_DIR, RESULTS_DIR]:
        d.mkdir(parents=True, exist_ok=True)

CA_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch'
for sub in ['', 'parf', 'multixi', 'scaleup', 'sarf_mass_variant', 'energetic_minima']:
    d = str(CA_DIR / sub) if sub else str(CA_DIR)
    if d not in sys.path:
        sys.path.insert(0, d)

CKPT_PREFIX   = 'fock_dcvt_owt' + (f'_{_variant_tag}' if _variant_tag else '')
TOTAL_STEPS   = 100_000
CKPT_INTERVAL = 7_500
CKPT_STEPS    = list(range(CKPT_INTERVAL, TOTAL_STEPS + 1, CKPT_INTERVAL))

print(f'CKPT_DIR    = {CKPT_DIR}')
print(f'RESULTS_DIR = {RESULTS_DIR}')
print(f'Steps: {TOTAL_STEPS:,}  checkpoints at: {CKPT_STEPS}')

IN_COLAB = True
Mounted at /content/drive
$ git clone --depth 1 --branch main https://github.com/dimitarpg13/semsimula-paper.git /content/semsimula-paper
$ pip install -q transformers huggingface_hub pyarrow
CKPT_DIR    = /content/drive/MyDrive/semsimula_fock_depthcond_vtheta_owt_xi5long_topk16_dt32da16_mh4_dcvt5x8_ob_untied_wsd_e5c/checkpoints
RESULTS_DIR = /content/drive/MyDrive/semsimula_fock_depthcond_vtheta_owt_xi5long_topk16_dt32da16_mh4_dcvt5x8_ob_untied_wsd_e5c/results
Steps: 100,000  checkpoints at: [7500, 15000, 22500, 30000, 37500, 45000, 52500, 60000, 67500, 75000, 82500, 90000, 97500]


In [3]:
# ── Cell 2: Checkpoint resolution + resume detection ─────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}  ({props.total_memory/1e9:.1f} GB)')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

resume_step = 0
resume_ckpt = None

for s in sorted(CKPT_STEPS, reverse=True):
    cand = CKPT_DIR / f'{CKPT_PREFIX}_step{s}.pt'
    if cand.exists():
        resume_ckpt = cand
        resume_step = s
        break

import glob as _glob, re as _re
_best_candidates = []
_canonical = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'
if _canonical.exists():
    _best_candidates.append(_canonical)
for _f in sorted(CKPT_DIR.glob(f'{CKPT_PREFIX}_step*_best.pt')):
    _best_candidates.append(_f)

_best_path = None
_best_step_found = resume_step
for _cand in _best_candidates:
    try:
        _bd = torch.load(_cand, map_location='cpu', weights_only=False)
        _s = _bd.get('step', 0)
        _p = _bd.get('val_ppl', float('inf'))
        del _bd
        if _s > _best_step_found:
            _best_step_found = _s
            _best_path = _cand
            _best_ppl = _p
            print(f'  Found best candidate: {_cand.name} (step {_s:,}, PPL {_p:.2f})')
    except Exception as e:
        print(f'[warn] could not inspect {_cand.name}: {e}')

if _best_path is not None and _best_step_found > resume_step:
    print(f'Best checkpoint (step {_best_step_found:,}, PPL {_best_ppl:.2f}) is more recent '
          f'than latest periodic checkpoint (step {resume_step:,}) — resuming from best.')
    resume_ckpt = _best_path
    resume_step = _best_step_found

if resume_ckpt is not None:
    print(f'\nResuming from: {resume_ckpt.name}  (step {resume_step:,})')
    print(f'Remaining: {TOTAL_STEPS - resume_step:,} steps')
else:
    print('No checkpoint found — training from scratch.')
    print(f'Total: {TOTAL_STEPS:,} steps  Checkpoints every {CKPT_INTERVAL:,}')

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition  (102.0 GB)
No checkpoint found — training from scratch.
Total: 100,000 steps  Checkpoints every 7,500


In [4]:
# ── Cell 3: Data loading (reuse cached OpenWebText) ──────────────
from data_module import get_batch

MAX_TRAIN_TOKENS = 1_000_000_000
VAL_TOKENS       = 2_000_000
CHUNK_SIZE       = 50_000
VOCAB_SIZE       = 50257

from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('gpt2')

train_cache = DATA_DIR / f'openwebtext_train_{MAX_TRAIN_TOKENS // 1_000_000}M.npy'
val_cache   = DATA_DIR / f'openwebtext_val_{VAL_TOKENS // 1_000_000}M.npy'

for alt_name in [
    # E2 run dir: lets the e5c / e5a arms reuse the tokenised OWT cache
    # instead of re-downloading when they get their own run directory.
    'semsimula_fock_depthcond_vtheta_owt_xi5long_topk16_dt32da16_mh4_dcvt5x8_ob_untied_wsd',
    'semsimula_fock_structured_vtheta_owt_phase4',
    'semsimula_fock_gaussian_sarf_openwebtext_phase5',
    'semsimula_splm_openwebtext_phase4',
    'semsimula_splm_openwebtext_scaleup',
    'semsimula_parf_multixi_openwebtext_scaleup',
    'semsimula_fock_multixi_openwebtext_scaleup',
    'semsimula_splm_openwebtext',
    'semsimula_fock_multihead_openwebtext',
    'semsimula_fock_multicontext_vtheta_owt',
]:
    if train_cache.exists():
        break
    if IN_COLAB:
        alt = Path(f'/content/drive/MyDrive/{alt_name}/data')
    else:
        alt = Path.home() / alt_name / 'data'
    alt_train = alt / f'openwebtext_train_{MAX_TRAIN_TOKENS // 1_000_000}M.npy'
    alt_val   = alt / f'openwebtext_val_{VAL_TOKENS // 1_000_000}M.npy'
    if alt_train.exists():
        import shutil
        print(f'Reusing data cache from {alt}')
        shutil.copy2(str(alt_train), str(train_cache))
        shutil.copy2(str(alt_val), str(val_cache))
        break

if train_cache.exists() and val_cache.exists():
    print('Loading cached OpenWebText tokens ...')
    train_ids = np.load(str(train_cache))
    val_ids   = np.load(str(val_cache))
    print(f'  train: {len(train_ids):,} tokens')
    print(f'  val:   {len(val_ids):,} tokens')
else:
    from datasets import load_dataset
    print(f'Streaming OpenWebText (target: {MAX_TRAIN_TOKENS:,} train + {VAL_TOKENS:,} val tokens) ...')
    ds = load_dataset('Skylion007/openwebtext', split='train', streaming=True, trust_remote_code=True)
    all_ids = []
    total = 0
    target = MAX_TRAIN_TOKENS + VAL_TOKENS
    chunk_texts = []
    n_docs = 0
    t0 = time.time()
    for example in ds:
        chunk_texts.append(example['text'])
        n_docs += 1
        if len(chunk_texts) >= CHUNK_SIZE:
            joined = '\n\n'.join(chunk_texts)
            chunk_ids = tok.encode(joined)
            all_ids.extend(chunk_ids)
            total = len(all_ids)
            elapsed = time.time() - t0
            print(f'  {n_docs:,} docs  {total:,} tokens  ({elapsed:.0f}s)', flush=True)
            chunk_texts = []
            del joined, chunk_ids
            if total >= target:
                break
    if chunk_texts:
        joined = '\n\n'.join(chunk_texts)
        all_ids.extend(tok.encode(joined))
        del joined, chunk_texts
    all_ids = np.array(all_ids, dtype=np.uint16)
    total = len(all_ids)
    print(f'Total streamed: {total:,} tokens from {n_docs:,} documents ({time.time() - t0:.0f}s)')
    val_ids   = all_ids[-VAL_TOKENS:]
    train_ids = all_ids[:-VAL_TOKENS]
    if len(train_ids) > MAX_TRAIN_TOKENS:
        train_ids = train_ids[:MAX_TRAIN_TOKENS]
    del all_ids
    np.save(str(train_cache), train_ids)
    np.save(str(val_cache), val_ids)
    print(f'  Cached: train={len(train_ids):,} -> {train_cache}')
    print(f'  Cached: val={len(val_ids):,}   -> {val_cache}')

print(f'train: {len(train_ids):,}   val: {len(val_ids):,}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Reusing data cache from /content/drive/MyDrive/semsimula_fock_depthcond_vtheta_owt_xi5long_topk16_dt32da16_mh4_dcvt5x8_ob_untied_wsd/data
Loading cached OpenWebText tokens ...
  train: 1,000,000,000 tokens
  val:   2,000,000 tokens
train: 1,000,000,000   val: 2,000,000


In [5]:
# ── Cell 4b: PMI Spectral Diversity — K_MIX estimator ──────────────
RANK_EFF_TINYSTORIES = None


def compute_pmi_effective_rank(token_ids, vocab_size, top_v=8192, window=5, n_components=512):
    """Compute effective spectral rank of the PMI matrix (Roy-Vetterli)."""
    print(f'  Building co-occurrence matrix (top_v={top_v}, window={window}) ...')
    token_counts = np.bincount(token_ids.astype(np.int64), minlength=vocab_size)
    top_v_ids = np.argsort(-token_counts)[:top_v]
    id_to_local = np.full(vocab_size, -1, dtype=np.int64)
    id_to_local[top_v_ids] = np.arange(top_v)

    cooc = np.zeros((top_v, top_v), dtype=np.float64)
    local_ids = id_to_local[token_ids.astype(np.int64)]
    for offset in range(1, window + 1):
        a, b = local_ids[:-offset], local_ids[offset:]
        valid = (a >= 0) & (b >= 0)
        np.add.at(cooc, (a[valid], b[valid]), 1.0)
    cooc = cooc + cooc.T

    row_sums = cooc.sum(axis=1, keepdims=True)
    total = cooc.sum()
    expected = row_sums * row_sums.T / total
    with np.errstate(divide='ignore', invalid='ignore'):
        pmi = np.log(cooc / np.maximum(expected, 1e-12))
    pmi = np.nan_to_num(pmi, nan=0.0, posinf=0.0, neginf=-20.0)
    np.fill_diagonal(pmi, 0.0)

    print(f'  Computing truncated SVD (k={n_components}) ...')
    from scipy.sparse.linalg import svds
    from scipy.sparse import csr_matrix
    pmi_sparse = csr_matrix(pmi)
    _, s, _ = svds(pmi_sparse, k=min(n_components, top_v - 1))
    s = np.sort(s)[::-1]

    s_norm = s / s.sum()
    s_norm = s_norm[s_norm > 1e-12]
    entropy = -np.sum(s_norm * np.log(s_norm))
    rank_eff = np.exp(entropy)
    return rank_eff, s


#print('Computing PMI spectral diversity for OpenWebText ...')
#t_pmi = time.time()
#rank_eff_owt, sv_owt = compute_pmi_effective_rank(train_ids, VOCAB_SIZE)
#print(f'  Done in {time.time() - t_pmi:.1f}s')
#print(f'\n  rank_eff(OpenWebText) = {rank_eff_owt:.1f}')
#print(f'  Top-5 singular values: {sv_owt[:5]}')
#print(f'  SV decay ratio (s[0]/s[50]): {sv_owt[0]/sv_owt[min(50, len(sv_owt)-1)]:.1f}')
#
#if RANK_EFF_TINYSTORIES is not None:
#    diversity_ratio = rank_eff_owt / RANK_EFF_TINYSTORIES
#    suggested_k = max(8, round(8 * diversity_ratio))
#    print(f'\n  rank_eff(TinyStories) = {RANK_EFF_TINYSTORIES:.1f}  (reference)')
#    print(f'  Diversity ratio: {diversity_ratio:.2f}x')
#    print(f'  Suggested V_THETA_WELLS_PER_HEAD = {suggested_k}')
#else:
#    print(f'\n  [info] RANK_EFF_TINYSTORIES not set.')
#    print(f'  Set it at the top of this cell for scaling guidance.')

In [6]:
# ── Cell 4: Model config + multi-context V_theta ─────────────────
import math
from model_fock_parf_multixi import FockMultiXiPARFLM, FockMultiXiPARFConfig
import model_fock_parf_v2
import model_parf_multixi
import model_parf
import model_parf_sparse

_XI_PRESETS = {
    5:       [0.25, 0.50, 0.75, 0.95, 0.99],
    '5long': [0.50, 0.75, 0.95, 0.99, 0.995],
    6:       [0.25, 0.50, 0.75, 0.95, 0.99, 0.995],
    '4long': [0.50, 0.75, 0.95, 0.995],
}
if XI_OVERRIDE is None:
    XI_ALPHA_INITS = [0.25, 0.50, 0.75, 0.95]
elif XI_OVERRIDE in _XI_PRESETS:
    XI_ALPHA_INITS = _XI_PRESETS[XI_OVERRIDE]
else:
    raise ValueError(f'Unsupported XI_OVERRIDE={XI_OVERRIDE!r}; use None, 5, 6, "5long", or "4long"')
XI_CHANNELS = len(XI_ALPHA_INITS)
print(f'Xi: {XI_CHANNELS} channels, alphas={XI_ALPHA_INITS}')
print(f'  Horizons: ~{[round(1/(1-a),1) for a in XI_ALPHA_INITS]} tokens')

LAMBDA_V       = 1e-2
BLOCK_SIZE     = 512

LOGFREQ_PATH = CA_DIR / 'scaleup' / 'results' / 'logfreq_surprisal_openwebtext.npy'
DRIVE_LOGFREQ = RESULTS_DIR / 'logfreq_surprisal_openwebtext.npy'

if LOGFREQ_PATH.exists():
    LOGFREQ_FILE = LOGFREQ_PATH
elif DRIVE_LOGFREQ.exists():
    LOGFREQ_FILE = DRIVE_LOGFREQ
else:
    counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE).astype(np.float64)
    p = (counts + 1.0) / (counts.sum() + VOCAB_SIZE)
    surprisal = (-np.log(p)).astype(np.float32)
    LOGFREQ_FILE = DRIVE_LOGFREQ
    LOGFREQ_FILE.parent.mkdir(parents=True, exist_ok=True)
    np.save(LOGFREQ_FILE, surprisal)

print(f'Logfreq: {LOGFREQ_FILE}')

ARCH_TIERS = [
    (384, 16, 32),
    (384, 12, 16),
    (256, 16, 16),
    (256,  8, 16),
]


def make_config(d, L, n_registers):
    return FockMultiXiPARFConfig(
        vocab_size=VOCAB_SIZE, d=d, max_len=1024,
        L=L, v_hidden=1024, v_depth=3, dt=1.0,
        mass_mode='logfreq',
        logfreq_path=str(LOGFREQ_FILE),
        logfreq_init_alpha=0.1,
        init_gamma=1.0,
        fixed_gamma=0.30,
        causal_force=True,
        ln_after_step=True,
        xi_channels=XI_CHANNELS,
        xi_alpha_inits=XI_ALPHA_INITS,
        xi_learnable=True,
        xi_alpha_init_mode='explicit',
        v_phi_kind=V_PHI_KIND,
        v_phi_d_type=V_PHI_D_TYPE,
        v_phi_d_angle=V_PHI_D_ANGLE,
        v_phi_eps=0.1,
        v_phi_phi_hidden=128,
        v_phi_theta_hidden=128,
        v_phi_mlp_hidden=V_PHI_MLP_HIDDEN,
        top_k=TOP_K,
        v_phi_n_heads=V_PHI_N_HEADS,
        use_output_bias=USE_OUTPUT_BIAS,
        tie_embeddings=TIE_EMBEDDINGS,
        score_head_hidden=32,
        gumbel_tau_init=1.0,
        gumbel_tau_min=0.3,
        gumbel_noise=True,
        use_gathered_v_phi=True,
        use_layer_checkpoint=True,
        ln_before_distance=True,
        per_layer_v_phi_scale=True,
        fock_version='v2',
        n_registers=n_registers,
        register_salience_decay=0.5,
        register_salience_threshold=0.005,
        creation_gate_hidden=64,
        stack_discipline=True,
        d_k=64,
        tau_create_init=8.0,
        reverse_channel=REVERSE_CHANNEL,
        reverse_channel_stable=REVERSE_CHANNEL_STABLE,
        reverse_channel_pre_ln=REVERSE_CHANNEL_PRE_LN,
        reverse_channel_soft_norm=REVERSE_CHANNEL_SOFT_NORM,
        reverse_channel_warmup_steps=REVERSE_CHANNEL_WARMUP_STEPS,
        per_register_tau=True,
        per_register_keys=True,
        ortho_register_init=True,
    )


def build_structured_vtheta(model, d, variant, device):
    """Swap model.V_theta with a structured variant.

    When V_THETA_N_HEADS > 1, uses MultiContextGaussianVTheta: one
    independent Gaussian well bank per xi channel.
    When V_THETA_N_HEADS == 1, falls back to the concat baseline
    (GaussianVThetaMultiXiAdapter) for comparison.
    """
    if variant == 'mlp':
        return
    xi_d = XI_CHANNELS * d
    from model_gaussian_vtheta import (
        MixtureGaussianVTheta, SARFGaussianVTheta,
        GaussianVThetaMultiXiAdapter, MultiContextGaussianVTheta,
        DepthConditionedMultiContextGaussianVTheta, install_depth_routing,
    )

    if variant == 'gaussian':
        _init_log_prec = -math.log(d)
        _prec_max = 2.0 / d

        if V_THETA_N_HEADS > 1 and V_THETA_DEPTH_CONDITION:
            # Option A: one shared multi-context bank + per-layer depth codes.
            n_layers = model.cfg.L
            model.V_theta = DepthConditionedMultiContextGaussianVTheta(
                d=d, K=V_THETA_WELLS_PER_HEAD, n_ctx=V_THETA_N_HEADS,
                n_layers=n_layers,
                w_scale=W_SCALE,
                init_log_precision=_init_log_prec,
                precision_max=_prec_max,
                code_init_std=V_THETA_DEPTH_CODE_INIT_STD,
            ).to(device)
            # Broadcast the active layer index to the shared bank on every
            # per-layer step (safe under gradient checkpointing).
            install_depth_routing(model)
            _n_code = model.V_theta.depth_code.numel()
            _sigma_eff_init = 1.0 / (math.exp(_init_log_prec) ** 0.5)
            _sigma_min = 1.0 / (_prec_max ** 0.5)
            print(f'V_theta -> DepthConditionedMultiContextGaussian('
                  f'{V_THETA_N_HEADS} heads x {V_THETA_WELLS_PER_HEAD} wells, '
                  f'L={n_layers} layers, code_params={_n_code:,}, '
                  f'w_scale={W_SCALE}, sigma_eff_init={_sigma_eff_init:.2f}, '
                  f'sigma_min={_sigma_min:.2f})')
            print(f'  depth routing installed on _fock_layer_step')
        elif V_THETA_N_HEADS > 1:
            model.V_theta = MultiContextGaussianVTheta(
                d=d, K=V_THETA_WELLS_PER_HEAD, n_ctx=V_THETA_N_HEADS,
                w_scale=W_SCALE,
                init_log_precision=_init_log_prec,
                precision_max=_prec_max,
            ).to(device)
            _sigma_eff_init = 1.0 / (math.exp(_init_log_prec) ** 0.5)
            _sigma_min = 1.0 / (_prec_max ** 0.5)
            print(f'V_theta -> MultiContextGaussian({V_THETA_N_HEADS} heads x '
                  f'{V_THETA_WELLS_PER_HEAD} wells = {V_THETA_N_HEADS * V_THETA_WELLS_PER_HEAD} total, '
                  f'w_scale={W_SCALE}, sigma_eff_init={_sigma_eff_init:.2f}, '
                  f'sigma_min={_sigma_min:.2f})')
        else:
            inner = MixtureGaussianVTheta(
                d=d, K=V_THETA_WELLS_PER_HEAD, w_scale=W_SCALE, xi_d=xi_d,
                init_log_precision=_init_log_prec,
                precision_max=_prec_max,
            )
            model.V_theta = GaussianVThetaMultiXiAdapter(
                inner, K=XI_CHANNELS, d=d,
            ).to(device)
            _sigma_eff_init = 1.0 / (math.exp(_init_log_prec) ** 0.5)
            _sigma_min = 1.0 / (_prec_max ** 0.5)
            print(f'V_theta -> ConcatGaussian(K={V_THETA_WELLS_PER_HEAD}, w_scale={W_SCALE}, '
                  f'sigma_eff_init={_sigma_eff_init:.2f}, sigma_min={_sigma_min:.2f})')

    elif variant == 'sarf':
        WINDOW = 5
        TOP_V = 8192
        token_counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE)
        top_v_ids = np.argsort(-token_counts)[:TOP_V]
        id_to_local = np.full(VOCAB_SIZE, -1, dtype=np.int64)
        id_to_local[top_v_ids] = np.arange(TOP_V)

        cooc = np.zeros((TOP_V, TOP_V), dtype=np.float64)
        local_ids = id_to_local[train_ids.astype(np.int64)]
        for offset in range(1, WINDOW + 1):
            a = local_ids[:-offset]
            b = local_ids[offset:]
            valid = (a >= 0) & (b >= 0)
            np.add.at(cooc, (a[valid], b[valid]), 1.0)
        cooc = cooc + cooc.T

        row_sums = cooc.sum(axis=1, keepdims=True)
        total = cooc.sum()
        expected = row_sums * row_sums.T / total
        with np.errstate(divide='ignore', invalid='ignore'):
            pmi = np.log(cooc / np.maximum(expected, 1e-12))
        pmi = np.nan_to_num(pmi, nan=0.0, posinf=0.0, neginf=-20.0)

        np.fill_diagonal(pmi, -np.inf)
        pmi_peaks = pmi.max(axis=1)
        anchor_local_ids = np.argsort(-pmi_peaks)[:SARF_N_ANCHORS]
        anchor_token_ids = top_v_ids[anchor_local_ids]
        anchor_positions = model.E.weight.data[anchor_token_ids].detach().clone()
        print(f'SARF anchors: {SARF_N_ANCHORS} PMI-peak tokens selected')
        print(f'  PMI peak range: [{pmi_peaks[anchor_local_ids[-1]]:.2f}, '
              f'{pmi_peaks[anchor_local_ids[0]]:.2f}]')

        _log_sigma_max = 0.5 * math.log(d) + 1.0
        _init_log_sigma = math.log(d) / 2.0

        inner = SARFGaussianVTheta(
            d=d, anchor_positions=anchor_positions, xi_d=xi_d, w_scale=W_SCALE,
            init_log_sigma=_init_log_sigma,
            log_sigma_max=_log_sigma_max,
        )
        model.V_theta = GaussianVThetaMultiXiAdapter(inner, K=XI_CHANNELS, d=d).to(device)

        with torch.no_grad():
            a = model.V_theta.inner.anchors
            a = (a - a.mean(dim=-1, keepdim=True)) / (a.std(dim=-1, keepdim=True) + 1e-5)
            model.V_theta.inner.anchors.copy_(a)

        print(f'V_theta -> SARF Gaussian(N_S={SARF_N_ANCHORS})')
        print(f'  Anchors re-normalised: norm={a.norm(dim=-1).mean():.2f}')
        print(f'  sigma_init={model.V_theta.inner.sigma.mean():.2f}')

    elif variant == 'sq3':
        from model_structured_vtheta import MixtureQuadraticVTheta
        from model_structured_vtheta_multixi import StructuredVThetaMultiXiAdapter

        inner = MixtureQuadraticVTheta(
            d=d, K=V_THETA_WELLS_PER_HEAD, tau=SQ3_TAU, init_a_bias=0.0,
            xi_d=xi_d,
        )

        if SQ3_CURV_MAX is not None:
            _orig_components = inner._components
            def _clamped_components(xi, _fn=_orig_components, _cmax=SQ3_CURV_MAX):
                mu, a, log_pi = _fn(xi)
                return mu, a.clamp(max=_cmax), log_pi
            inner._components = _clamped_components

        model.V_theta = StructuredVThetaMultiXiAdapter(
            inner, K=XI_CHANNELS, d=d,
        ).to(device)

        print(f'V_theta -> SQ3 Mixture(K={V_THETA_WELLS_PER_HEAD}, tau={SQ3_TAU})')
        print(f'  xi_d={xi_d}, curvature_max={SQ3_CURV_MAX}')
        n_sq3 = sum(p.numel() for p in model.V_theta.parameters())
        print(f'  SQ3 params: {n_sq3:,}')

    else:
        raise ValueError(f'Unknown V_theta variant: {variant}')


# ── Try architecture tiers ─────────────────────────────────────────
model = None
model_cfg = None
for d, L, M in ARCH_TIERS:
    try:
        cfg = make_config(d, L, M)
        mdl = FockMultiXiPARFLM(cfg).to(DEVICE)
        n_v_theta_mlp = sum(p.numel() for p in mdl.V_theta.parameters())
        build_structured_vtheta(mdl, d, V_THETA_VARIANT, DEVICE)
        n = mdl.num_params()
        n_v_theta = sum(p.numel() for p in mdl.V_theta.parameters())
        _vt_label = {'mlp': 'MLP', 'sq3': 'SQ3', 'gaussian': 'Gaussian', 'sarf': 'SARF'}[V_THETA_VARIANT]
        _mc_label = f' ({V_THETA_N_HEADS} heads)' if V_THETA_N_HEADS > 1 else ' (concat)'
        print(f'Trying d={d} L={L} M={M} -> {n:,} params '
              f'(V_theta {n_v_theta_mlp:,} MLP -> {n_v_theta:,} {_vt_label}{_mc_label})')
        if DEVICE == 'cuda':
            _rng = np.random.default_rng(42)
            _xb, _yb = get_batch(train_ids, 2, BLOCK_SIZE, _rng)
            _x = torch.from_numpy(_xb).to(DEVICE)
            _y = torch.from_numpy(_yb).to(DEVICE)
            _, _loss = mdl(_x, _y)
            _loss.backward()
            mdl.zero_grad(set_to_none=True)
            del _x, _y, _xb, _yb, _loss
            torch.cuda.empty_cache()
            print(f'OOM probe passed (batch=2)')
        model = mdl
        model_cfg = cfg
        break
    except RuntimeError as e:
        if 'out of memory' in str(e).lower():
            print(f'  OOM at d={d} L={L} M={M} — trying next tier ...')
            del mdl
            gc.collect()
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()
            continue
        raise

if model is None:
    raise RuntimeError('All architecture tiers OOMed.')

if USE_OUTPUT_BIAS:
    _ob_counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE)
    model.init_output_bias_from_logfreq(_ob_counts)
    print(f'Output bias <- log-unigram-freq  '
          f'(b range [{model.out_bias.min().item():.2f}, '
          f'{model.out_bias.max().item():.2f}])')

# ── Auto batch size (GRAD_ACCUM is fixed from Cell 0) ──────────────
# Probe list is GPU-aware: H100 (>=80 GB) tries up to 16; A100 (40 GB)
# skips 16 and tries 12 before falling back to 8/6/4.
BATCH_SIZE = 4
if DEVICE == 'cuda':
    _vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    _probe_sizes = [16, 12, 8, 6, 4] if _vram_gb >= 70 else [12, 8, 6, 4]
    for bs in _probe_sizes:
        try:
            _rng = np.random.default_rng(42)
            _xb, _yb = get_batch(train_ids, bs, BLOCK_SIZE, _rng)
            _x = torch.from_numpy(_xb).to(DEVICE)
            _y = torch.from_numpy(_yb).to(DEVICE)
            _, _loss = model(_x, _y)
            _loss.backward()
            model.zero_grad(set_to_none=True)
            del _x, _y, _xb, _yb, _loss
            torch.cuda.empty_cache()
            BATCH_SIZE = bs
            print(f'Auto batch: {bs} x accum={GRAD_ACCUM} (eff={bs*GRAD_ACCUM})')
            break
        except RuntimeError:
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()
            continue

EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM
n_params = model.num_params()
n_v_theta = sum(p.numel() for p in model.V_theta.parameters())
IS_STRUCTURED = V_THETA_VARIANT in ('gaussian', 'sarf', 'sq3')

_vtheta_name = {'gaussian': 'Gaussian', 'sarf': 'SARF', 'sq3': 'SQ3', 'mlp': 'MLP'}[V_THETA_VARIANT]
print(f'\nModel: FockMultiXiPARFLM v2.1 + {_vtheta_name} V_theta')
print(f'  params: {n_params:,}  (V_theta: {n_v_theta:,})')
print(f'  d={model_cfg.d}  L={model_cfg.L}  M={model_cfg.n_registers}')
print(f'  V_theta={V_THETA_VARIANT}  lambda_V={LAMBDA_V}')
if V_THETA_N_HEADS > 1:
    print(f'  V_theta heads: {V_THETA_N_HEADS} x {V_THETA_WELLS_PER_HEAD} wells = '
          f'{V_THETA_N_HEADS * V_THETA_WELLS_PER_HEAD} total attractors')
    if V_THETA_DEPTH_CONDITION:
        _routed = getattr(model, '_depth_routing_installed', False)
        print(f'  depth-conditioned: shared bank across L={model_cfg.L} layers '
              f'+ per-layer codes  (routing installed={_routed})')
else:
    print(f'  V_theta concat baseline: K={V_THETA_WELLS_PER_HEAD}')
print(f'  V_phi={V_PHI_KIND} x {V_PHI_N_HEADS} head(s)  top_k={TOP_K}  '
      f'd_type={V_PHI_D_TYPE}  d_angle={V_PHI_D_ANGLE}')
print(f'  batch={BATCH_SIZE} x accum={GRAD_ACCUM} (eff={EFFECTIVE_BATCH})')
print(f'  IS_STRUCTURED={IS_STRUCTURED}')


Xi: 5 channels, alphas=[0.5, 0.75, 0.95, 0.99, 0.995]
  Horizons: ~[2.0, 4.0, 20.0, 100.0, 200.0] tokens
Logfreq: /content/drive/MyDrive/semsimula_fock_depthcond_vtheta_owt_xi5long_topk16_dt32da16_mh4_dcvt5x8_ob_untied_wsd_e5c/results/logfreq_surprisal_openwebtext.npy
V_theta -> DepthConditionedMultiContextGaussian(5 heads x 8 wells, L=16 layers, code_params=30,720, w_scale=1.0, sigma_eff_init=19.60, sigma_min=13.86)
  depth routing installed on _fock_layer_step
Trying d=384 L=16 M=32 -> 53,378,060 params (V_theta 4,460,545 MLP -> 11,873,320 Gaussian (5 heads))
OOM probe passed (batch=2)
Output bias <- log-unigram-freq  (b range [-20.72, -3.30])
Auto batch: 16 x accum=2 (eff=32)

Model: FockMultiXiPARFLM v2.1 + Gaussian V_theta
  params: 53,378,060  (V_theta: 11,873,320)
  d=384  L=16  M=32
  V_theta=gaussian  lambda_V=0.01
  V_theta heads: 5 x 8 wells = 40 total attractors
  depth-conditioned: shared bank across L=16 layers + per-layer codes  (routing installed=True)
  V_phi=structura

In [ ]:
# ── Cell 5: Training loop ─────────────────────────────────────────

LR            = 3e-4      # higher peak LR (WSD keeps it here longer)
WEIGHT_DECAY  = 0.01
WARMUP_STEPS  = int(WSD_WARMUP_FRAC * TOTAL_STEPS) if LR_SCHEDULE == 'wsd' else 4000
GRAD_CLIP     = 1.0
GRAD_CLIP_VPHI = 0.3

# ── Per-group (per-layer) gradient clipping ──────────────────────
# When True, each top-level module's gradients are clipped to their
# OWN max-norm instead of a single global rescale.  This stops one
# exploding component (e.g. the Fock registers or one V_phi head)
# from either dominating the global norm (zeroing every useful
# gradient) or slipping under it.  Substring matches in
# GRAD_CLIP_OVERRIDES take priority over the per-module default.
PER_GROUP_CLIP = True
GRAD_CLIP_OVERRIDES = {
    'V_phi': GRAD_CLIP_VPHI,   # pairwise potential: keep the tight 0.3 clip
    'creation_gate': 0.3,      # Fock QKV creation gate (W_Q / W_K / log_tau)
    'destruction_gate': 0.3,   # Fock destruction gates
    # The reverse-channel GATE is a single global scalar whose gradient is a
    # sum over B*T*L positions, so it is naturally O(1e3+) even when the
    # (RMS-normed, warmup-gated) force it controls is ~0.  Give it its own
    # group (matched BEFORE 'reverse_ch') and exclude it from the watchdog
    # total below, so this benign scalar can't trigger spurious reloads.
    'reverse_channel_scale': 0.1,  # Fock reverse-channel gate scalar (benign)
    'reverse_ch': 0.1,         # Fock reverse-channel projections + LN + logit_scale
    'register': 0.3,           # register_embed (Fock register bank)
    'depth_code': 0.5,         # per-layer depth codes
}
# Groups counted for diagnostics/clipping but EXCLUDED from the global
# pre-clip norm that drives the spike debugger and watchdog.  Both Fock
# reverse-channel groups are tightly per-group clipped (<=0.1), so their
# UPDATE is bounded regardless of pre-clip norm.  Their large pre-clip
# gradient is a benign artifact: the stabilised readout RMS-normalises the
# force (Qf_rms=1.0) and the warmup-gated scale is ~1e-7 early, so the
# forward force is tiny -- but the 1/||Q|| Jacobian of the output norm,
# acting on a small natural force (r_rms~0.04), inflates the PRE-clip
# projection gradient into the thousands.  That is clipped away before the
# optimiser sees it, so it is not a stability signal and must not drive the
# watchdog (same rationale as reverse_channel_scale).
WATCHDOG_EXCLUDE_GROUPS = {'override:reverse_channel_scale', 'override:reverse_ch'}

# ── Per-step gradient-spike debugger ─────────────────────────────
# Fires the instant a step's PRE-clip total grad-norm crosses the
# threshold (not only at LOG_INTERVAL or on a sustained watchdog
# breach), printing the per-group breakdown so the exact culprit of a
# transient spike is captured at the step it happens.
GRAD_SPIKE_DEBUG     = True
GRAD_SPIKE_THRESHOLD = 100.0   # pre-clip total norm that counts as a spike
GRAD_SPIKE_COOLDOWN  = 0       # min steps between spike prints (0 = print every spike)
EVAL_INTERVAL = 500    # PPL update every ~500 steps (~25-80 min depending on step speed)
EVAL_ITERS    = 40
LOG_INTERVAL  = 50     # training loss line every ~50 steps (~3-8 min)
SEED          = 0

if V_THETA_VARIANT == 'sq3':
    LR        = 8e-5
    GRAD_CLIP = 0.5
    print(f'[SQ3 override] LR={LR}, GRAD_CLIP={GRAD_CLIP}')

# Resolve WSD_LR_FLOOR now that LR is final.
if WSD_LR_FLOOR is None:
    WSD_LR_FLOOR = LR * 0.05

GRAD_NORM_EMA_ALPHA = 0.05
GRAD_NORM_EMA_THRESHOLD = 50.0
GRAD_NORM_EMA_PATIENCE = 200

torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)


def lr_schedule(step):
    """Unified LR schedule supporting cosine and WSD."""
    if LR_SCHEDULE == 'wsd':
        warmup_end = int(WSD_WARMUP_FRAC * TOTAL_STEPS)
        stable_end = int((WSD_WARMUP_FRAC + WSD_STABLE_FRAC) * TOTAL_STEPS)
        if step < warmup_end:
            return LR * (step + 1) / max(warmup_end, 1)
        elif step < stable_end:
            return LR
        else:
            decay_steps = TOTAL_STEPS - stable_end
            progress = (step - stable_end) / max(decay_steps, 1)
            cos_decay = 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))
            return WSD_LR_FLOOR + (LR - WSD_LR_FLOOR) * cos_decay
    else:
        if step < WARMUP_STEPS:
            return LR * (step + 1) / WARMUP_STEPS
        progress = (step - WARMUP_STEPS) / max(TOTAL_STEPS - WARMUP_STEPS, 1)
        return LR * 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))


def forward_with_vreg(x, targets, lambda_v):
    h0 = model._embed(x)
    h_L, _ = model._stack_forward(h0, x, return_trajectory=False)
    logits = model.compute_logits(h_L)
    loss_ntp = F.cross_entropy(
        logits.reshape(-1, model_cfg.vocab_size),
        targets.reshape(-1),
    )
    v_reg_value = torch.tensor(0.0, device=x.device)
    if lambda_v > 0:
        xis = model.xi_module(h_L.detach())
        V_vals = model.V_theta(xis, h_L)
        if V_THETA_VARIANT == 'sq3':
            v_reg_value = torch.log1p(V_vals ** 2).mean()
        else:
            v_reg_value = (V_vals ** 2).mean()
        if BG_QUAD_EPS > 0:
            bg = BG_QUAD_EPS * (h_L ** 2).sum(dim=-1, keepdim=True).mean()
            loss = loss_ntp + lambda_v * v_reg_value + bg
        else:
            loss = loss_ntp + lambda_v * v_reg_value
    else:
        loss = loss_ntp
    return loss, loss_ntp, v_reg_value


@torch.no_grad()
def evaluate():
    model.eval()
    losses = []
    for _ in range(EVAL_ITERS):
        xb, yb = get_batch(val_ids, BATCH_SIZE, BLOCK_SIZE, rng)
        x = torch.from_numpy(xb).to(DEVICE)
        y = torch.from_numpy(yb).to(DEVICE)
        with torch.enable_grad():
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))


def save_checkpoint(step_num, val_loss_val, tag_suffix=''):
    ckpt = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optim.state_dict(),
        'model_cfg': asdict(model_cfg),
        'train_cfg': {
            'batch_size': BATCH_SIZE, 'block_size': BLOCK_SIZE,
            'grad_accum': GRAD_ACCUM, 'effective_batch': EFFECTIVE_BATCH,
            'steps': TOTAL_STEPS, 'lr': LR, 'weight_decay': WEIGHT_DECAY,
            'warmup_steps': WARMUP_STEPS, 'grad_clip': GRAD_CLIP,
            'grad_clip_vphi': GRAD_CLIP_VPHI,
            'optimizer': OPTIMIZER, 'grad_centralization': GRAD_CENTRALIZATION,
            'lambda_v': LAMBDA_V, 'v_theta_variant': V_THETA_VARIANT,
            'lr_schedule': LR_SCHEDULE,
            'v_theta_n_heads': V_THETA_N_HEADS,
            'v_theta_wells_per_head': V_THETA_WELLS_PER_HEAD,
            'v_theta_depth_condition': V_THETA_DEPTH_CONDITION,
            'v_theta_depth_code_init_std': V_THETA_DEPTH_CODE_INIT_STD,
        },
        'step': step_num,
        'val_loss': val_loss_val,
        'val_ppl': math.exp(val_loss_val),
        'gamma': model.gamma.item(),
        'xi_alphas': model.xi_alpha_values(),
        'variant': (
            f'fock_parf_multixi_v2.1_{V_THETA_VARIANT}_'
            + (f'dcvt{V_THETA_N_HEADS}' if (V_THETA_N_HEADS > 1 and V_THETA_DEPTH_CONDITION)
               else f'mcvt{V_THETA_N_HEADS}')
        ),
        'corpus': 'openwebtext',
        'phase': 6,
        'seed': SEED,
    }
    fname = f'{CKPT_PREFIX}_step{step_num}{tag_suffix}.pt'
    path = CKPT_DIR / fname
    for _attempt in range(2):
        try:
            torch.save(ckpt, path)
            break
        except OSError as _e:
            if _e.errno == 107 and _attempt == 0:
                print(f'[WARN] Drive transport error saving checkpoint; remounting... ({_e})')
                try:
                    from google.colab import drive as _drv
                    _drv.mount('/content/drive', force_remount=True)
                except Exception as _re:
                    print(f'[WARN] Drive remount failed: {_re}')
                    print(f'[WARN] Checkpoint NOT saved: {path}')
                    return None
            else:
                print(f'[WARN] Checkpoint save failed: {_e}')
                return None
    print(f'  Checkpoint saved: {path}  (PPL={math.exp(val_loss_val):.2f})')
    if '_best' in tag_suffix:
        canonical = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'
        import shutil
        shutil.copy2(path, canonical)
        print(f'  Canonical best: {canonical}')
    return path


# ── Optimizer ──
_trainable = [p for p in model.parameters() if p.requires_grad]
if OPTIMIZER == 'adamw':
    optim = torch.optim.AdamW(_trainable, lr=LR,
                              weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95))
elif OPTIMIZER == 'lamb':
    try:
        import torch_optimizer
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torch_optimizer'])
        import torch_optimizer
    optim = torch_optimizer.Lamb(_trainable, lr=LR,
                                weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95))
elif OPTIMIZER == 'lion':
    try:
        from lion_pytorch import Lion
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lion-pytorch'])
        from lion_pytorch import Lion
    optim = Lion(_trainable, lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.99))
else:
    raise ValueError(f'Unknown OPTIMIZER={OPTIMIZER!r}; choose adamw / lamb / lion')
print(f'Optimizer: {type(optim).__name__}')

# ── Resume ──
if resume_ckpt is not None and resume_step < TOTAL_STEPS:
    print(f'Resuming from checkpoint at step {resume_step:,}: {resume_ckpt}')
    ckpt_data = torch.load(resume_ckpt, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt_data['model_state_dict'], strict=False)
    # E5c transitional reset (see REVERSE_CHANNEL_RESET_SCALE above): zero
    # the gate learned under the old unnormalised readout so the stabilised
    # force opens cleanly from scratch under the warmup ramp.
    if (REVERSE_CHANNEL and REVERSE_CHANNEL_STABLE and REVERSE_CHANNEL_RESET_SCALE
            and getattr(model, 'reverse_channel_scale', None) is not None):
        with torch.no_grad():
            model.reverse_channel_scale.zero_()
            if hasattr(model, 'reverse_warmup_step'):
                model.reverse_warmup_step.zero_()
        print('  [E5c] reverse_channel_scale re-zeroed + warmup reset for clean stable start')
    if 'optimizer_state_dict' in ckpt_data:
        try:
            optim.load_state_dict(ckpt_data['optimizer_state_dict'])
            print('  Optimizer state restored.')
        except (ValueError, KeyError) as e:
            print(f'  [info] Optimizer state incompatible, starting fresh: {e}')
    prev_ppl = ckpt_data.get('val_ppl', float('nan'))
    print(f'  Model loaded. Previous PPL: {prev_ppl:.2f}')
    del ckpt_data
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

# ── Training state ──
log_path = RESULTS_DIR / 'training_log.jsonl'
_log_fh = [log_path.open('a')]

def _log_write(record_str):
    """Write a JSONL record to the Drive log, remounting on transport error."""
    for _attempt in range(2):
        try:
            _log_fh[0].write(record_str)
            _log_fh[0].flush()
            return
        except OSError as _e:
            if _e.errno == 107 and _attempt == 0:
                print(f'[WARN] Drive transport error on log write; remounting... ({_e})')
                try:
                    from google.colab import drive as _drv
                    _drv.mount('/content/drive', force_remount=True)
                    try:
                        _log_fh[0].close()
                    except Exception:
                        pass
                    _log_fh[0] = log_path.open('a')
                except Exception as _re:
                    print(f'[WARN] Drive remount failed: {_re}; log record lost, training continues.')
                    return
            else:
                print(f'[WARN] Log write failed (attempt {_attempt+1}): {_e}; training continues.')
                return

log_f = _log_fh

t0 = time.time()
model.train()
run_ntp = 0.0
run_vreg = 0.0
n_run = 0
n_skipped = 0

best_val_ppl = float('inf')
_best_ckpt_path = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'

if not _best_ckpt_path.exists():
    _step_bests = sorted(CKPT_DIR.glob(f'{CKPT_PREFIX}_step*_best.pt'))
    if _step_bests:
        _best_ckpt_path = _step_bests[-1]
        print(f'No canonical _best.pt; using {_best_ckpt_path.name}')
        import shutil
        _canonical = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'
        shutil.copy2(_best_ckpt_path, _canonical)
        _best_ckpt_path = _canonical
        print(f'  Copied to canonical: {_canonical.name}')

if _best_ckpt_path.exists():
    try:
        _bd = torch.load(_best_ckpt_path, map_location='cpu', weights_only=False)
        best_val_ppl = _bd.get('val_ppl', float('inf'))
        print(f'Restored running best PPL: {best_val_ppl:.2f}')
        del _bd
    except Exception as e:
        print(f'[warn] {e}')

_grad_norm_ema = 0.0
_grad_norm_above_thresh = 0

def _reload_best():
    if not _best_ckpt_path.exists():
        return resume_step
    ckpt = torch.load(_best_ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'], strict=False)
    try:
        optim.load_state_dict(ckpt['optimizer_state_dict'])
    except (ValueError, KeyError):
        pass
    s = ckpt.get('step', 0)
    p = ckpt.get('val_ppl', float('nan'))
    del ckpt
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    print(f'[watchdog] Reloaded best: step {s:,} PPL {p:.2f}')
    return s

steps_this_session = 0

# ── Schedule summary ──
if LR_SCHEDULE == 'wsd':
    _warmup_end = int(WSD_WARMUP_FRAC * TOTAL_STEPS)
    _stable_end = int((WSD_WARMUP_FRAC + WSD_STABLE_FRAC) * TOTAL_STEPS)
    _sched_str = (f'WSD: warmup 0->{_warmup_end:,}, stable {_warmup_end:,}->{_stable_end:,}, '
                  f'decay {_stable_end:,}->{TOTAL_STEPS:,}, floor={WSD_LR_FLOOR:.2e}')
else:
    _sched_str = f'cosine: warmup {WARMUP_STEPS:,} steps'

_vt_mode = ('Depth-conditioned multi-context'
            if (V_THETA_N_HEADS > 1 and V_THETA_DEPTH_CONDITION)
            else ('Multi-context' if V_THETA_N_HEADS > 1 else 'Concat'))
print(f'\n{"="*60}')
print(f'{_vt_mode} V_theta ({_vtheta_name}): steps {resume_step+1:,} -> {TOTAL_STEPS:,}')
print(f'  batch={BATCH_SIZE} x accum={GRAD_ACCUM} (eff={EFFECTIVE_BATCH})')
print(f'  block={BLOCK_SIZE}  lr={LR}  grad_clip={GRAD_CLIP}  grad_clip_vphi={GRAD_CLIP_VPHI}')
print(f'  schedule: {_sched_str}')
print(f'  d={model_cfg.d}  L={model_cfg.L}  M={model_cfg.n_registers}')
print(f'  V_theta={V_THETA_VARIANT}  heads={V_THETA_N_HEADS}  wells/head={V_THETA_WELLS_PER_HEAD}  params={n_params:,}')
if V_THETA_N_HEADS > 1 and V_THETA_DEPTH_CONDITION:
    print(f'  depth-conditioned: shared bank + per-layer codes '
          f'(routing installed={getattr(model, "_depth_routing_installed", False)})')
print(f'  watchdog: threshold={GRAD_NORM_EMA_THRESHOLD} patience={GRAD_NORM_EMA_PATIENCE}')
if PER_GROUP_CLIP:
    print(f'  per-group clip: default={GRAD_CLIP}  overrides={GRAD_CLIP_OVERRIDES}')
if REVERSE_CHANNEL:
    _rev_mode = ('stable (QK-norm + '
                 + ('soft-norm' if REVERSE_CHANNEL_SOFT_NORM else 'RMS-norm')
                 + (' + pre-LN' if REVERSE_CHANNEL_PRE_LN else '') + ')'
                 ) if REVERSE_CHANNEL_STABLE else 'vanilla'
    print(f'  reverse channel: {_rev_mode}  warmup={REVERSE_CHANNEL_WARMUP_STEPS} forwards')
else:
    print('  reverse channel: OFF (E5a ablation)')
print(f'{"="*60}\n')


def _assign_clip_group(pname):
    """Map a parameter name to (group_key, max_norm).

    Override substrings win (so every V_phi / Fock tensor is clipped as
    one tight group); otherwise the parameter is grouped by its
    top-level module and clipped to the default GRAD_CLIP.
    """
    low = pname.lower()
    for sub, thr in GRAD_CLIP_OVERRIDES.items():
        if sub.lower() in low:
            return f'override:{sub}', thr
    return pname.split('.', 1)[0], GRAD_CLIP


def per_group_grad_norms(model):
    """Non-mutating snapshot of pre-clip per-group gradient norms.

    Used by the spike debugger so the culprit breakdown reflects the
    gradients BEFORE any clipping rescales them.
    """
    groups = {}
    for n, p in model.named_parameters():
        if not p.requires_grad or p.grad is None:
            continue
        key, _ = _assign_clip_group(n)
        groups.setdefault(key, []).append(p)
    out = {}
    for key, ps in groups.items():
        sq = 0.0
        for p in ps:
            sq += float(p.grad.detach().norm()) ** 2
        out[key] = sq ** 0.5
    return out


def clip_grads_per_group(model):
    """Clip gradients independently per module group.

    Returns (total_norm_tensor, per_group_norms_dict).  total_norm is the
    global pre-clip norm (sqrt of summed per-group squared norms), so the
    existing watchdog threshold stays directly comparable.
    """
    groups, thr = {}, {}
    _dev = None
    for n, p in model.named_parameters():
        if not p.requires_grad or p.grad is None:
            continue
        if _dev is None:
            _dev = p.grad.device
        key, mx = _assign_clip_group(n)
        groups.setdefault(key, []).append(p)
        thr[key] = mx
    total_sq = torch.zeros((), device=_dev) if _dev is not None else torch.zeros(())
    per_group = {}
    for key, ps in groups.items():
        gn = nn.utils.clip_grad_norm_(ps, thr[key])
        per_group[key] = float(gn)
        if key not in WATCHDOG_EXCLUDE_GROUPS:
            total_sq = total_sq + gn.detach() ** 2
    return total_sq.sqrt(), per_group


import inspect
print('exclude set        :', WATCHDOG_EXCLUDE_GROUPS)
print('scale in overrides :', 'reverse_channel_scale' in GRAD_CLIP_OVERRIDES)
print('exclusion in clip  :', 'WATCHDOG_EXCLUDE_GROUPS' in inspect.getsource(clip_grads_per_group))
print('stable / pre_ln    :', model.reverse_ch.stable, model.reverse_ch.pre_ln)
print('gate value         :', float(model.reverse_channel_scale))
for n, _ in model.named_parameters():
    if 'reverse' in n.lower():
        print(' ', _assign_clip_group(n)[0], '<-', n)

## -----------TEST CODE SHOULD BE COMMENTED
import torch
model.train()
with torch.no_grad():
    model.reverse_channel_scale.zero_()
    model.reverse_warmup_step.zero_()

xb, yb = get_batch(train_ids, BATCH_SIZE, BLOCK_SIZE, rng)
x = torch.from_numpy(xb).to(DEVICE); y = torch.from_numpy(yb).to(DEVICE)
model.zero_grad(set_to_none=True)
loss, ntp, vreg = forward_with_vreg(x, y, LAMBDA_V)
loss.backward()

g = {}
for n, p in model.named_parameters():
    if p.grad is None:
        continue
    k, _ = _assign_clip_group(n)
    g[k] = g.get(k, 0.0) + float(p.grad.norm()) ** 2
print("gate =", float(model.reverse_channel_scale.detach()))
for k, v in sorted(g.items(), key=lambda kv: -kv[1])[:8]:
    print(f"  {k:35s} {v**0.5:10.3f}")

import model_fock_parf_multixi, model_fock_parf_v2
print("multixi file:", model_fock_parf_multixi.__file__)
print("v2 file     :", model_fock_parf_v2.__file__)
print("ReverseChannel forward has 'rms':",
      "rms" in __import__("inspect").getsource(model.reverse_ch.forward))
##### _______TEST CODE SHOULD BE COMMENTED


#### _____MORE_TEST CODE FOR GRADIENT EXPLOSION

import torch

# clean E5c state
with torch.no_grad():
    model.reverse_channel_scale.zero_()
    model.reverse_warmup_step.zero_()

# capture ||Q_force|| and ||r|| from inside the reverse channel
caps = {}
_orig = model.reverse_ch.forward
def _hooked(h_tokens, r_active, active_mask):
    out = _orig(h_tokens, r_active, active_mask)
    caps['Qf_rms']  = float(out.pow(2).mean().sqrt())
    caps['r_rms']   = float(r_active.pow(2).mean().sqrt())
    caps['h_rms']   = float(h_tokens.pow(2).mean().sqrt())
    return out
model.reverse_ch.forward = _hooked

model.zero_grad(set_to_none=True)
for _ in range(GRAD_ACCUM):
    xb, yb = get_batch(train_ids, BATCH_SIZE, BLOCK_SIZE, rng)
    x = torch.from_numpy(xb).to(DEVICE); y = torch.from_numpy(yb).to(DEVICE)
    loss, ntp, vreg = forward_with_vreg(x, y, LAMBDA_V)
    (loss / GRAD_ACCUM).backward()

model.reverse_ch.forward = _orig  # restore

warm = min(1.0, int(model.reverse_warmup_step) / max(1, REVERSE_CHANNEL_WARMUP_STEPS))
scale = float(torch.tanh(model.reverse_channel_scale.detach())) * warm
print(f"gate={float(model.reverse_channel_scale.detach()):.3e}  warmup_step={int(model.reverse_warmup_step)}  warm={warm:.4f}  effective scale={scale:.3e}")
print("forward magnitudes:", {k: round(v, 4) for k, v in caps.items()})
print("--- per-parameter pre-clip grad norms (reverse_ch + scale) ---")
for n, p in model.named_parameters():
    if ('reverse_ch' in n or 'reverse_channel_scale' in n) and p.grad is not None:
        print(f"  {n:35s} {float(p.grad.norm()):14.3f}")


#### _____MORE TEST CODE FOR GRADIENT EXPLOSION


_last_pg_norms = {}
_last_spike_step = -10**9
for step in range(resume_step, TOTAL_STEPS):
    lr_now = lr_schedule(step)
    for g in optim.param_groups:
        g['lr'] = lr_now

    optim.zero_grad(set_to_none=True)
    accum_ntp = 0.0
    accum_vreg = 0.0
    for _acc in range(GRAD_ACCUM):
        xb, yb = get_batch(train_ids, BATCH_SIZE, BLOCK_SIZE, rng)
        x = torch.from_numpy(xb).to(DEVICE)
        y = torch.from_numpy(yb).to(DEVICE)
        loss, loss_ntp, v_reg = forward_with_vreg(x, y, LAMBDA_V)
        (loss / GRAD_ACCUM).backward()
        accum_ntp  += loss_ntp.item()       / GRAD_ACCUM
        accum_vreg += float(v_reg.detach()) / GRAD_ACCUM

    if GRAD_CENTRALIZATION:
        for p in model.parameters():
            if p.grad is not None and p.grad.dim() >= 2:
                p.grad.sub_(p.grad.mean(dim=tuple(range(1, p.grad.dim())), keepdim=True))

    if PER_GROUP_CLIP:
        grad_norm, _last_pg_norms = clip_grads_per_group(model)
    else:
        _last_pg_norms = per_group_grad_norms(model) if GRAD_SPIKE_DEBUG else {}
        if model.V_phi is not None:
            nn.utils.clip_grad_norm_(model.V_phi.parameters(), GRAD_CLIP_VPHI)
        grad_norm = nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], GRAD_CLIP,
        )

    # ── per-step gradient-spike debugger ──
    if GRAD_SPIKE_DEBUG:
        _tot_preclip = float(grad_norm)
        if (_tot_preclip > GRAD_SPIKE_THRESHOLD
                and (step - _last_spike_step) >= GRAD_SPIKE_COOLDOWN):
            _last_spike_step = step
            if _last_pg_norms:
                _top = sorted(_last_pg_norms.items(),
                              key=lambda kv: kv[1], reverse=True)[:8]
                _brk = '  '.join(f'{k}={v:.1f}' for k, v in _top)
            else:
                _brk = '(enable PER_GROUP_CLIP or GRAD_SPIKE_DEBUG for breakdown)'
            print(f'\n[spike] step {step+1}: pre-clip total grad={_tot_preclip:.1f}  '
                  f'ntp={accum_ntp:.3f}  v_reg={accum_vreg:.4f}')
            print(f'[spike]   top groups: {_brk}')
    if torch.isfinite(grad_norm) and math.isfinite(accum_ntp):
        optim.step()
        if IS_STRUCTURED and V_THETA_N_HEADS == 1:
            if hasattr(getattr(model.V_theta, 'inner', None), 'clamp_params'):
                model.V_theta.inner.clamp_params()
        elif IS_STRUCTURED and V_THETA_N_HEADS > 1:
            for bank in model.V_theta.banks:
                if hasattr(bank, 'clamp_params'):
                    bank.clamp_params()
    else:
        n_skipped += 1
        optim.zero_grad(set_to_none=True)

    # ── Watchdog ──
    _raw_gn = float(grad_norm)
    _grad_norm_ema = (1 - GRAD_NORM_EMA_ALPHA) * _grad_norm_ema + GRAD_NORM_EMA_ALPHA * _raw_gn
    if _grad_norm_ema > GRAD_NORM_EMA_THRESHOLD:
        _grad_norm_above_thresh += 1
    else:
        _grad_norm_above_thresh = 0

    if _grad_norm_above_thresh >= GRAD_NORM_EMA_PATIENCE:
        print(f'\n[watchdog] EMA grad_norm={_grad_norm_ema:.1f} > {GRAD_NORM_EMA_THRESHOLD} '
              f'for {_grad_norm_above_thresh} steps at step {step+1}.')
        if _last_pg_norms:
            _top = sorted(_last_pg_norms.items(), key=lambda kv: kv[1], reverse=True)[:5]
            print('[watchdog] top group norms (pre-clip): '
                  + ', '.join(f'{k}={v:.1f}' for k, v in _top))
        _reload_best()
        _grad_norm_ema = 0.0
        _grad_norm_above_thresh = 0
        n_skipped += 1

    run_ntp += accum_ntp
    run_vreg += accum_vreg
    n_run += 1
    steps_this_session += 1

    if (step + 1) % LOG_INTERVAL == 0:
        avg_ntp = run_ntp / n_run
        avg_vreg = run_vreg / n_run
        run_ntp, run_vreg, n_run = 0.0, 0.0, 0
        elapsed = time.time() - t0
        sec_per_step = elapsed / steps_this_session
        remaining = (TOTAL_STEPS - step - 1) * sec_per_step
        alphas = model.xi_alpha_values()
        alpha_str = ','.join(f'{a:.3f}' for a in alphas)
        _top_grp = ''
        if PER_GROUP_CLIP and _last_pg_norms:
            _k, _v = max(_last_pg_norms.items(), key=lambda kv: kv[1])
            _top_grp = f'top[{_k}]={_v:.1f}  '
        print(
            f'step {step+1:7d}/{TOTAL_STEPS}  '
            f'ntp={avg_ntp:.4f}  v_reg={avg_vreg:.4f}  lr={lr_now:.2e}  '
            f'grad={float(grad_norm):.2f}  {_top_grp}gamma={model.gamma.item():.3f}  '
            f'alpha=[{alpha_str}]  '
            f'{elapsed:.0f}s  (~{remaining/3600:.1f}h remaining)'
        )
        _log_write(json.dumps({
            'step': step + 1, 'train_loss': avg_ntp, 'v_reg': avg_vreg,
            'lr': lr_now, 'grad_norm': float(grad_norm),
            'gamma': model.gamma.item(), 'xi_alphas': alphas,
            'elapsed_sec': elapsed, 'sec_per_step': sec_per_step,
        }) + '\n')

    if (step + 1) % EVAL_INTERVAL == 0:
        val_loss = evaluate()
        val_ppl = math.exp(val_loss)
        is_best = val_ppl < best_val_ppl
        if is_best:
            best_val_ppl = val_ppl
        elapsed = time.time() - t0
        marker = '*** NEW BEST ***' if is_best else ''
        print(f'>>> EVAL step {step+1:,}  val_loss={val_loss:.4f}  '
              f'val_ppl={val_ppl:.2f}  best={best_val_ppl:.2f}  '
              f'{marker}  ({elapsed:.0f}s)')
        _log_write(json.dumps({
            'step': step + 1, 'val_loss': val_loss,
            'val_ppl': val_ppl, 'best_ppl': best_val_ppl,
        }) + '\n')
        if is_best:
            save_checkpoint(step + 1, val_loss, tag_suffix='_best')

    if (step + 1) in set(CKPT_STEPS):
        if (step + 1) % EVAL_INTERVAL != 0:
            val_loss = evaluate()
            val_ppl = math.exp(val_loss)
        save_checkpoint(step + 1, val_loss)

_log_fh[0].close()
print(f'\nTraining complete. Best PPL: {best_val_ppl:.2f}')

Optimizer: AdamW

Depth-conditioned multi-context V_theta (Gaussian): steps 1 -> 100,000
  batch=16 x accum=2 (eff=32)
  block=512  lr=0.0003  grad_clip=1.0  grad_clip_vphi=0.3
  schedule: WSD: warmup 0->5,000, stable 5,000->65,000, decay 65,000->100,000, floor=1.50e-05
  d=384  L=16  M=32
  V_theta=gaussian  heads=5  wells/head=8  params=53,378,060
  depth-conditioned: shared bank + per-layer codes (routing installed=True)
  watchdog: threshold=50.0 patience=200
  per-group clip: default=1.0  overrides={'V_phi': 0.3, 'creation_gate': 0.3, 'destruction_gate': 0.3, 'reverse_channel_scale': 0.1, 'reverse_ch': 0.1, 'register': 0.3, 'depth_code': 0.5}
  reverse channel: stable (QK-norm + soft-norm + pre-LN)  warmup=4000 forwards

exclude set        : {'override:reverse_ch', 'override:reverse_channel_scale'}
scale in overrides : True
exclusion in clip  : True
stable / pre_ln    : True True
gate value         : 0.0
  override:reverse_channel_scale <- reverse_channel_scale
  override:reverse_

/tmp/ipykernel_754/2715942506.py:437: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print('gate value         :', float(model.reverse_channel_scale))


gate = 0.0
  lm_head                                  0.376
  E                                        0.357
  P                                        0.154
  out_bias                                 0.014
  V_theta                                  0.003
  xi_module                                0.001
  override:depth_code                      0.000
  override:V_phi                           0.000
multixi file: /content/semsimula-paper/notebooks/conservative_arch/parf/model_fock_parf_multixi.py
v2 file     : /content/semsimula-paper/notebooks/conservative_arch/parf/model_fock_parf_v2.py
ReverseChannel forward has 'rms': True
gate=0.000e+00  warmup_step=2  warm=0.0005  effective scale=0.000e+00
forward magnitudes: {'Qf_rms': 0.1471, 'r_rms': 0.0016, 'h_rms': 0.9937}
--- per-parameter pre-clip grad norms (reverse_ch + scale) ---
  reverse_channel_scale                        0.000
  reverse_ch.logit_scale                       0.000
  reverse_ch.W_Q_rev.weight                    0.000
